In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("projects")
    .getOrCreate()
)

NOTE: Picked up JDK_JAVA_OPTIONS: --add-modules=jdk.incubator.vector
NOTE: Picked up JDK_JAVA_OPTIONS: --add-modules=jdk.incubator.vector
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/31 12:38:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/31 12:38:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from pyspark.sql import functions as f

projects_bronze = spark.read.parquet("/app/data/bronze/tabProyecto")

customers = spark.read.parquet("/app/data/silver/customers")

join = projects_bronze.join(customers, on=projects_bronze.cliente == customers.name, how="inner")

projects = join.select(
    f.expr("uuid()").alias("id"),
    projects_bronze["name"].alias("external_id"),
    projects_bronze["creation"].alias("created_at"),
    projects_bronze["modified"].alias("updated_at"),
    projects_bronze["margen_total"].alias("margin"),
    customers["id"].alias("customer_id")
)

In [3]:
projects.write.mode("overwrite").parquet("/app/data/silver/projects")